# LSST ↔ DeepLense Data Pipeline — End-to-End Walkthrough

This notebook demonstrates the complete pipeline from querying the Rubin
Science Platform (RSP) catalog to producing a PyTorch DataLoader ready
for any DeepLense task (lens finding, classification, super-resolution).

## Contents
1. Setup & authentication
2. Query the DP0.2 object catalog via TAP
3. Retrieve calibrated image cutouts via SIA v2
4. Preprocess: asinh normalisation + resize to 64×64
5. Wrap in a `RubinLensDataset` (offline / online modes)
6. Feed into a DeepLense ResNet-18 classifier
7. Visualise sample cutouts

> **Authentication note** — Sections 2 & 3 require a Rubin Science Platform
> personal access token.  Obtain one at https://data.lsst.cloud/  
> Set `RSP_TOKEN` in your environment or paste it in the cell below.
> Sections 4–7 work fully **offline** using the bundled mock data.

In [ ]:
# ── 0. Install dependencies (uncomment if needed) ────────────────────────────
# !pip install pyvo astropy torch torchvision pyyaml pandas matplotlib

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add parent directory to path (when running from notebooks/)
sys.path.insert(0, str(Path('..').resolve()))

from lsst_deeplense import (
    RubinTAPClient,
    RubinSIAClient,
    Normaliser,
    CutoutExtractor,
    RubinLensDataset,
)
from lsst_deeplense.preprocessing import build_deeplense_transforms
from lsst_deeplense.utils import load_config

print('All imports OK')

## 1. Load pipeline configuration

In [ ]:
cfg = load_config('../configs/pipeline_config.yaml')
print('Search centre:', cfg['data']['search_ra'], cfg['data']['search_dec'])
print('Bands:', cfg['preprocessing']['bands'])
print('Output size:', cfg['preprocessing']['output_size'], 'px')

## 2. TAP catalog query

We query the Rubin DP0.2 DC2 simulated catalog for photometric
objects brighter than _i_ = 24 and with signal-to-noise > 10.
These are our **lens candidates**.

This is a real ADQL cone-search against the public RSP endpoint.
The DC2 field covers ~300 deg² of sky simulation.

In [ ]:
# Set your RSP token here OR via env: export RSP_TOKEN=...
token = os.environ.get('RSP_TOKEN', '')

tap = RubinTAPClient(token=token)

candidates = tap.query_lens_candidates(
    ra=cfg['data']['search_ra'],
    dec=cfg['data']['search_dec'],
    radius_deg=cfg['data']['search_radius_deg'],
    band=cfg['data']['band'],
    mag_limit=cfg['data']['mag_limit'],
    snr_min=cfg['data']['snr_min'],
)

print(f'Retrieved {len(candidates):,} candidates')
candidates.head()

## 3. Image cutout retrieval via SIA v2

For each candidate we request 10-arcsecond cutouts in _g_, _r_, _i_
bands from the DP0.2 deep coadd images.

**Physics note**: 10 arcsec ≈ 50 px at LSST's 0.2"/px scale, capturing
the Einstein radius of typical massive galaxy lenses (~1–3") with
ample background for sky estimation.

In [ ]:
sia = RubinSIAClient(token=token)

# Demonstrate with the first candidate
demo = candidates.iloc[0]
print(f'Fetching cutouts for objectId={demo.get("objectId", "?")}  '
      f'(ra={demo.ra:.4f}, dec={demo.dec:.4f})')

band_arrays = sia.fetch_cutouts(
    ra=demo.ra,
    dec=demo.dec,
    size_arcsec=cfg['preprocessing']['size_arcsec'],
    bands=cfg['preprocessing']['bands'],
)

print('Retrieved bands:', list(band_arrays.keys()))
for b, arr in band_arrays.items():
    print(f'  {b}: shape={arr.shape}  min={arr.min():.2f}  max={arr.max():.2f}')

## 4. Preprocessing

### 4a. Asinh normalisation

The asinh stretch is the standard in astronomy for images where you need
to preserve both bright lens-galaxy nuclei **and** faint lensing arcs
simultaneously.  The linear regime at small fluxes (controlled by `a`)
ensures faint arc structure is not compressed into noise.

In [ ]:
norm = Normaliser(strategy='asinh', asinh_a=0.1)

# Normalise each band
normed = {b: norm(arr) for b, arr in band_arrays.items()}

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for col, band in enumerate(['g', 'r', 'i']):
    if band not in band_arrays:
        continue
    axes[0, col].imshow(band_arrays[band], cmap='viridis', origin='lower')
    axes[0, col].set_title(f'{band}-band (raw)')
    axes[1, col].imshow(normed[band], cmap='viridis', origin='lower')
    axes[1, col].set_title(f'{band}-band (asinh norm)')

for ax in axes.ravel():
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_normalisation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Normalised pixel range:', 
      {b: (v.min().round(3), v.max().round(3)) for b, v in normed.items()})

### 4b. Resize to DeepLense standard (64×64) and stack

In [ ]:
extractor = CutoutExtractor(output_size=cfg['preprocessing']['output_size'])
stacked = extractor.stack_bands(normed, band_order=('g', 'r', 'i'))

print('Stacked tensor shape (C, H, W):', stacked.shape)  # → (3, 64, 64)
print('dtype:', stacked.dtype)

## 5. RubinLensDataset

### Offline mode — using pre-cached `.npy` files

In production you would call `sia.fetch_cutouts()` for all candidates
and cache them to disk.  Here we generate synthetic arrays to
demonstrate the Dataset interface without an RSP connection.

In [ ]:
import tempfile
import torch

# Create a synthetic cache directory
cache_dir = Path(tempfile.mkdtemp())

rng = np.random.default_rng(0)
mock_positions = [(150.1 + i*0.01, 2.2 + i*0.01) for i in range(20)]

for ra, dec in mock_positions:
    arr = rng.random((3, 64, 64)).astype(np.float32)
    np.save(str(cache_dir / f'{ra:.6f}_{dec:.6f}.npy'), arr)

# Labels CSV (0=no substructure, 1=subhalo, 2=vortex)
label_df = pd.DataFrame([
    {'ra': ra, 'dec': dec, 'label': i % 3}
    for i, (ra, dec) in enumerate(mock_positions)
])
label_csv = cache_dir / 'labels.csv'
label_df.to_csv(label_csv, index=False)

# Build dataset
transform = build_deeplense_transforms(image_size=64, is_train=True, num_channels=3)

dataset = RubinLensDataset.from_npy_dir(
    npy_dir=cache_dir,
    transform=transform,
    label_csv=label_csv,
)

print(f'Dataset size: {len(dataset)} samples')
img, label = dataset[0]
print(f'Sample shape: {img.shape}  label: {label}')

In [ ]:
# ── DataLoader ───────────────────────────────────────────────────────────────
loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
)

batch_imgs, batch_labels = next(iter(loader))
print(f'Batch shape: {batch_imgs.shape}  labels: {batch_labels}')

## 6. Quick sanity check with a DeepLense-style ResNet-18

In [ ]:
import torchvision.models as models
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

# 3-channel input → 3-class output (no-sub / subhalo / vortex)
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 3)
model = model.to(device)

with torch.no_grad():
    logits = model(batch_imgs.to(device))

print('Logit shape:', logits.shape)      # (8, 3)
print('Predicted classes:', logits.argmax(dim=1).cpu().numpy())

## 7. Visualise a batch of cutouts

In [ ]:
CLASS_NAMES = ['No Substructure', 'Subhalo', 'Vortex']

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i, ax in enumerate(axes.ravel()):
    img_i = batch_imgs[i].numpy()   # (3, 64, 64)
    # Display the i-band (channel 2) for clarity
    channel = img_i[2] if img_i.shape[0] == 3 else img_i[0]
    # Rescale [-1,1] → [0,1] for display
    display = (channel - channel.min()) / (channel.max() - channel.min() + 1e-8)
    ax.imshow(display, cmap='gray', origin='lower')
    ax.set_title(CLASS_NAMES[int(batch_labels[i])], fontsize=9)
    ax.axis('off')

plt.suptitle('LSST DP0.2 mock cutouts — i-band (64×64 px)', fontsize=12)
plt.tight_layout()
plt.savefig('sample_cutouts.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

| Step | Component | Output |
|------|-----------|--------|
| Catalog query | `RubinTAPClient` | `pd.DataFrame` of candidates |
| Image retrieval | `RubinSIAClient` | `dict[band → np.ndarray]` |
| Normalisation | `Normaliser(strategy='asinh')` | `np.ndarray` in [0,1] |
| Resize & stack | `CutoutExtractor` | `(C, 64, 64)` float32 |
| Dataset | `RubinLensDataset` | PyTorch `Dataset` |
| DataLoader | `torch.utils.data.DataLoader` | Batched tensors |

The `RubinLensDataset` is a **drop-in replacement** for DeepLense's
existing `LensDataset`: same interface, same transform pipeline,
same output tensor format — so all existing training scripts
work unchanged.